In [2]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.rates import COLS, yield_changes, fit_pca, pc_shock
from src.alm import (load_balance_sheet, delta_eve_curve, delta_nii_curve, equity)

yields = pd.read_csv("../data/yields.csv", index_col="date", parse_dates=True)
dy = yield_changes(yields)
print(dy.shape, dy.isna().sum().sum())

pca = fit_pca(dy)
print((pca.explained_variance_ratio_ * 100).round(1))

loadings = pd.DataFrame(pca.components_.T, index=COLS,
                        columns=["PC1 level", "PC2 slope", "PC3 curvature", "PC4", "PC5"])
loadings.round(3)

(2953, 5) 0
[93.1  5.3  0.7  0.6  0.4]


,PC1 level,PC2 slope,PC3 curvature,PC4,PC5
2yr,0.407,-0.583,-0.214,-0.627,0.234
3yr,0.420,-0.485,-0.196,0.697,-0.253
5yr,0.469,0.074,0.742,-0.181,-0.436
7yr,0.475,0.302,0.181,0.252,0.766
10yr,0.460,0.573,-0.576,-0.158,-0.323


In [3]:
bs = load_balance_sheet()
eq = equity(bs)

rows = []
for i, name in enumerate(["Level", "Slope", "Curvature"]):
    for n in (1, -1):
        s = pc_shock(pca, i, n_sigma=n)
        d = delta_eve_curve(bs, s)
        rows.append({"scenario": f"{name} {'+' if n > 0 else '-'}1 sigma",
                     "shock_2yr": s.loc[2], "shock_10yr": s.loc[10],
                     "delta_eve": d, "eve_pct_equity": d / eq * 100,
                     "delta_nii": delta_nii_curve(bs, s)})

pca_table = pd.DataFrame(rows).set_index("scenario")
pca_table.round(3)

Reading file: /Users/bimansaha/Projects/bank-rate-risk-model/model/balance_sheet.csv


,shock_2yr,shock_10yr,delta_eve,eve_pct_equity,delta_nii
scenario,,,,,
Level +1 sigma,0.681,0.769,-1.040,-12.997,0.071
Level -1 sigma,-0.681,-0.769,1.040,12.997,-0.071
Slope +1 sigma,-0.232,0.228,-0.232,-2.903,0.023
Slope -1 sigma,0.232,-0.228,0.232,2.903,-0.023
Curvature +1 sigma,-0.030,-0.081,-0.221,-2.761,0.012
Curvature -1 sigma,0.030,0.081,0.221,2.761,-0.012
